What is Pydantic?
Pydantic is a library that makes sure your data is exactly what you expect it to be.

In standard Python, we can add 'type hints' to suggest what data type a variable should be. But Python doesn't actually strict enforce it. Let's look at an example.

In [2]:
# Standard Python behavior
def play_game(score: int):
    print(f"Your score is {score}!")

# We said 'score' is an integer, but we give it a string.
# Python doesn't care, it just runs it anyway!
play_game("One Hundred")

Your score is One Hundred!


Enter Pydantic
Pydantic changes the rules. If you say something is an integer, Pydantic guarantees it is an integer. If it isn't, Pydantic will stop the program and tell you exactly what went wrong.

In [4]:
from pydantic import BaseModel

# We create a Pydantic 'Model' by inheriting from BaseModel
class Player(BaseModel):
    name: str
    score: int

# 1. Perfect Data
player1 = Player(name="Alex", score=50)
print("Player 1:", player1)
print("----")

# 2. "Smart" Data Conversion
# Notice how score is a string "100", but it contains a number.
player2 = Player(name="Sam", score="100")
print("Player 2:", player2)
print("Did Pydantic magically convert it to an integer?", type(player2.score))

Player 1: name='Alex' score=50
----
Player 2: name='Sam' score=100
Did Pydantic magically convert it to an integer? <class 'int'>


What happens when the data is completely wrong?
If we pass a word instead of a number, Pydantic will throw a ValidationError.

Run the cell below and look closely at the error message. It tells you exactly where the mistake is!

In [5]:
# 3. Bad Data
# This will crash because "Fifty" cannot be magically converted to the number 50.
player3 = Player(name="Max", score="Fifty")

ValidationError: 1 validation error for Player
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Fifty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

Making Robust Schemas
Sometimes, checking if something is an integer or string isn't enough.

What if we want an age to be greater than 13? What if we want a password to be very strong?

Let's look at a real-world example: A User Sign-Up system.

In [6]:
from pydantic import BaseModel, Field, field_validator

# ye code understand karne mai mujue problem thi

class UserSignUp(BaseModel):
    username: str = Field(min_length=3, max_length=15) # Must be between 3-15 characters
    age: int = Field(ge=13) # ge means "Greater than or Equal to"
    account_type: str = "free" # Default value if nothing is given doesn't even need Field()
    
    # We can write completely custom rules using @field_validator
    @field_validator('username')
    @classmethod
    def check_username_no_spaces(cls, value: str) -> str:
        if " " in value:
            raise ValueError("Usernames cannot contain spaces!")
        return value

In [7]:
user1 = UserSignUp(
    username="GamerKing99",
    age=18
)

# Notice how account_type automatically became "free"
print(user1)

username='GamerKing99' age=18 account_type='free'


Breaking the Built-in Rules (Field)
If we provide an age that is too young, Pydantic stops it. (Run the cell below and read the error!)

In [8]:
too_young_user = UserSignUp(
    username="Timmy",
    age=10  # Must be >= 13!
)

ValidationError: 1 validation error for UserSignUp
age
  Input should be greater than or equal to 13 [type=greater_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

Breaking the Custom Rules (@field_validator)
If we provide a username with spaces, our custom validator will catch it. (Run the cell below and read the custom error message we wrote!)

In [9]:
bad_name_user = UserSignUp(
    username="Cool Guy",
    age=25
)

ValidationError: 1 validation error for UserSignUp
username
  Value error, Usernames cannot contain spaces! [type=value_error, input_value='Cool Guy', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error